# Edge-2-Star Feasible Region And Symmetry Breaking

This notebook inspects the edge-2-star sample cloud, validates the known upper envelope, and visualizes the `e=1/2` stability scan against the reported crossover `tilde t ~= 0.03727637`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "graphon_space").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "outputs" / "final" / "data"
DATA

In [ ]:
samples = pd.read_parquet(DATA / "samples_2star.parquet")
boundary = pd.read_parquet(DATA / "boundary_2star.parquet")
stability = pd.read_parquet(DATA / "stability_2star_e05.parquet")

def two_star_upper(e):
    e = np.asarray(e)
    return np.where(np.isclose(e, 0.5), np.sqrt(2) / 4, np.where(e > 0.5, e**1.5, (1 - e)**1.5 + 2 * e - 1))

pd.Series({
    "sample rows": len(samples),
    "k min": int(samples["k"].min()),
    "k max": int(samples["k"].max()),
    "boundary rows": len(boundary),
    "stability rows": len(stability),
})

In [ ]:
curve = np.linspace(0, 1, 600)

fig, ax = plt.subplots(figsize=(8, 5.5))
sc = ax.scatter(samples["e"], samples["t_2star"], c=samples["k"], s=3, alpha=0.25, cmap="viridis", rasterized=True)
ax.plot(curve, curve**2, color="black", lw=1.6, label="ER / constant degree: t=e^2")
ax.plot(curve, two_star_upper(curve), color="#b2182b", lw=1.6, label="known upper envelope")
ax.set_xlabel("edge density e")
ax.set_ylabel("2-star density t")
ax.set_title("Edge-2-star sampled feasible region")
ax.legend(loc="upper left")
fig.colorbar(sc, ax=ax, label="k")
fig

In [ ]:
upper = boundary[boundary["direction"] == "max"].sort_values("target_e").copy()
lower = boundary[boundary["direction"] == "min"].sort_values("target_e").copy()
upper["abs_error_vs_upper"] = (upper["observed_t"] - upper["known_upper"]).abs()

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.plot(lower["target_e"], lower["observed_t"], "o-", ms=3, label="estimated lower")
ax.plot(upper["target_e"], upper["observed_t"], "o-", ms=3, label="estimated upper")
ax.plot(curve, two_star_upper(curve), "--", color="black", lw=1.4, label="known upper")
ax.plot(curve, curve**2, ":", color="gray", lw=1.4, label="ER / lower")
ax.set_xlabel("edge density e")
ax.set_ylabel("2-star density t")
ax.set_title("Fixed-edge boundary validation")
ax.legend()
display(pd.Series({
    "median upper-boundary abs error": upper["abs_error_vs_upper"].median(),
    "worst upper-boundary abs error": upper["abs_error_vs_upper"].max(),
}))
fig

In [ ]:
ok = stability[stability["success"]].copy()
idx = ok.groupby("target_t_tilde")["entropy"].idxmax()
winners = ok.loc[idx].sort_values("target_t_tilde")

fig, ax = plt.subplots(figsize=(8, 5))
for family, group in winners.groupby("family"):
    ax.plot(group["target_t_tilde"], group["entropy"], "o-", ms=4, label=family)
ax.axvline(0.03727637, color="#b2182b", ls="--", lw=1.4, label="reported crossover")
ax.set_xlabel("reduced 2-star density t - e^2 at e=1/2")
ax.set_ylabel("best entropy found")
ax.set_title("Edge-2-star e=1/2 stability scan")
ax.legend()
display(winners[["target_t_tilde", "family", "entropy", "symmetry_class"]])
fig